# Surface Energies of SrTiO3(011) and (001) Terminations

> **R. I. Eglitis and David Vanderbilt**
> *First-principles calculations of atomic and electronic structure of SrTiO3 (001) and (011) surfaces*
> Phys. Rev. B 77, 195408 (2008)
> [DOI: 10.1103/PhysRevB.77.195408](https://doi.org/10.1103/PhysRevB.77.195408)

Computes the cleavage, relaxation and surface energies of the seven-plane slabs created in the
[structure notebook](slab_strontium_titanate.ipynb), TiO-, Sr- and O-terminated (011) and SrO- and TiO2-terminated (001),
with the paper's Eqs. (1)-(5) applied to total energies, and compares them with its Table VII.

<h2 style="color:green">Usage</h2>

1. Create the materials in the [structure notebook](slab_strontium_titanate.ipynb), which saves the bulk cell and the slabs to the `uploads` folder.
1. Set the material names and parameters in cells 1.2-1.4 (or use the defaults); `RELAX = True` adds the relaxation and surface energies.
1. Click "Run" > "Run All" to run all cells.
1. Wait for the jobs to complete.
1. Scroll down to view the results.

## Summary

1. Set up the environment and parameters: install packages (JupyterLite only) and configure the material names, model and compute parameters.
1. Authenticate and initialize API client: authenticate via browser, initialize the client, then select account and project.
1. Load the bulk cell and the slabs by name, print provenance, and save them to the platform.
1. Configure the shared DFT model and one k-point density for every cell.
1. Configure compute: get the list of clusters and create a compute configuration.
1. Run one total energy job on each cell as cut.
1. Relax each slab at fixed cell if `RELAX`.
1. Retrieve the total energies and compute the cleavage, relaxation and surface energies.
1. Compare with Eglitis & Vanderbilt (2008), Table VII.

## 1. Set up the environment and parameters
### 1.1. Install packages (JupyterLite)

In [ ]:
from mat3ra.notebooks_utils.packages import install_packages

await install_packages("made|specific_examples|api_examples")


### 1.2. Material names

In [ ]:
# Names saved by slab_strontium_titanate.ipynb.
BULK_NAME = "SrTiO3, Strontium Titanate, CUB (Pm-3m) 3D (Bulk), mp-5229"
SLAB_NAMES = {
    "TiO": "SrTiO3(011) TiO-terminated 7 planes",
    "Sr": "SrTiO3(011) Sr-terminated 7 planes",
    "O": "SrTiO3(011) O-terminated 7 planes",
    "SrO": "SrTiO3(001) SrO-terminated 7 planes",
    "TiO2": "SrTiO3(001) TiO2-terminated 7 planes",
}
# Slabs that are stoichiometric together; they share the energy of cleaving the crystal equally (paper Sec. III.D).
CLEAVAGE_GROUPS = [("TiO", "Sr"), ("O",), ("SrO", "TiO2")]

### 1.3. Parameters

In [ ]:
from datetime import datetime
from mat3ra.ide.compute import QueueName

ORGANIZATION_NAME = None  # set to your organization name (full or partial); otherwise, your default one is used
FOLDER = "./uploads"

TOTAL_ENERGY_SEARCH_TERM = "total_energy.json"
RELAX_WORKFLOW_SEARCH_TERM = "fixed_cell_relaxation.json"
APPLICATION_NAME = "espresso"

# False: cleavage energies of the slabs as cut. True: also relaxes all atoms of each slab at fixed cell, for the
# relaxation and surface energies. paper: only the near-surface planes (two in Sec. III.B, three in Table VII).
RELAX = False

CLUSTER_NAME = None
QUEUE_NAME = QueueName.OR
PPN = 16  # OR allows 16 cores per job
TIME_LIMIT = "04:00:00"

timestamp = datetime.now().strftime("%Y-%m-%d %H:%M")
POLL_INTERVAL = 60  # seconds

### 1.4. DFT model parameters

In [ ]:
# paper: hybrid B3PW with Gaussian basis sets (CRYSTAL). Ours: PBE with GBRV ultrasoft plane waves, so absolute
# energies shift; the paper gives no PBE surface energies to compare with.
FUNCTIONAL = "pbe"
PSEUDOPOTENTIAL_TYPE = "us"
ECUTWFC = 40   # Ry, GBRV's recommended wavefunction cutoff
ECUTRHO = 200  # Ry, GBRV's recommended charge-density cutoff

# paper: 8x8 for every slab. Ours: one density for every cell, the grid it gives is printed per material.
KPOINT_DENSITY = 6.5  # Å⁻¹
MODEL_TAG = f"{FUNCTIONAL}-{PSEUDOPOTENTIAL_TYPE} {ECUTWFC}-{ECUTRHO}Ry k{KPOINT_DENSITY}"

SCF_UNIT = "pw_scf"
RELAX_UNIT = "pw_relax"

## 2. Authenticate and initialize API client
### 2.1. Authenticate

In [ ]:
from mat3ra.notebooks_utils.auth import authenticate

await authenticate()


### 2.2. Initialize API client

In [ ]:
from mat3ra.api_client import APIClient

client = APIClient.authenticate()
client


### 2.3. Select account

In [ ]:
client.list_accounts()


In [ ]:
selected_account = client.my_account

if ORGANIZATION_NAME:
    selected_account = client.get_account(name=ORGANIZATION_NAME)

ACCOUNT_ID = selected_account.id
print(f"✅ Selected account ID: {ACCOUNT_ID}, name: {selected_account.name}")


### 2.4. Select project

In [ ]:
projects = client.projects.list({"isDefault": True, "owner._id": ACCOUNT_ID})
project_id = projects[0]["_id"]
print(f"✅ Using project: {projects[0]['name']} ({project_id})")


## 3. Load the materials
### 3.1. Load from the uploads folder or the platform, and print provenance

In [ ]:
from collections import Counter
from mat3ra.notebooks_utils.core.entity.material.api import load_material

materials_by_name = {name: load_material(client, FOLDER, name, ACCOUNT_ID)
                     for name in (BULK_NAME, *SLAB_NAMES.values())}
for name, material in materials_by_name.items():
    a, b, c = material.lattice.a, material.lattice.b, material.lattice.c
    element_counts = sorted(Counter(material.basis.elements.values).items())
    composition = "".join(f"{element}{count}" for element, count in element_counts)
    print(f"{name}: {composition}, {material.basis.number_of_atoms} atoms, cell {a:.3f} x {b:.3f} x {c:.3f} Å")

### 3.2. Save the materials to the platform

In [ ]:
from mat3ra.notebooks_utils.core.entity.material.api import get_or_create_material

saved_materials = {name: get_or_create_material(client, material, ACCOUNT_ID)
                   for name, material in materials_by_name.items()}

## 4. Configure the shared DFT model and k-grid
### 4.1. DFT model

In [ ]:
from mat3ra.standata.applications import ApplicationStandata
from mat3ra.standata.model_tree import ModelTreeStandata
from mat3ra.ade.application import Application
from mat3ra.mode import ModelFactory

application_config = ApplicationStandata.get_by_name_first_match(APPLICATION_NAME)
application = Application(**application_config)

model_config = ModelTreeStandata.get_model_by_parameters(type="dft", subtype="gga", functional=FUNCTIONAL)
model_config["method"] = {"type": "pseudopotential", "subtype": PSEUDOPOTENTIAL_TYPE}
model = ModelFactory.create(model_config)
print(f"Using application: {application.name}, model: {MODEL_TAG}")


### 4.2. k-grid per material

In [ ]:
from mat3ra.notebooks_utils.workflow import kgrid_from_density

kgrid = {name: kgrid_from_density(material, KPOINT_DENSITY, periodic_dims=(0, 1, 2) if name == BULK_NAME else (0, 1))
         for name, material in materials_by_name.items()}
for name, grid in kgrid.items():
    print(f"{name}: k-grid {grid}")

## 5. Create the compute configuration
### 5.1. Select cluster

In [ ]:
clusters = client.clusters.list()
print(f"Available clusters: {[cluster['hostname'] for cluster in clusters]}")


### 5.2. Create the compute configuration for the jobs

In [ ]:
from mat3ra.ide.compute import Compute

if CLUSTER_NAME:
    cluster = next((candidate for candidate in clusters if CLUSTER_NAME in candidate["hostname"]), None)
    if cluster is None:
        raise ValueError(f"Cluster '{CLUSTER_NAME}' not found. "
                         f"Available: {[candidate['hostname'] for candidate in clusters]}")
else:
    cluster = clusters[0]
compute = Compute(cluster=cluster, queue=QUEUE_NAME, ppn=PPN, timeLimit=TIME_LIMIT)
print(f"Using cluster: {compute.cluster.hostname}, queue: {QUEUE_NAME}, ppn: {PPN}, "
      f"time limit: {TIME_LIMIT}")


## 6. Total energy of every cell as cut

In [ ]:
from mat3ra.standata.workflows import WorkflowStandata
from mat3ra.wode.workflows import Workflow
from mat3ra.notebooks_utils.workflow import apply_planewave_cutoffs, apply_scf_kgrid
from mat3ra.notebooks_utils.job import create_job
from mat3ra.notebooks_utils.core.entity.job.api import find_job_for_material


def find_or_create_job(name, workflow_config, workflow_label, unit_name):
    """Returns the id of this material's job with the shared model and k-grid, creating the job if none exists,
    and whether it was created."""
    workflow = Workflow.create(workflow_config)
    workflow.name = f"{workflow_label} {name} {MODEL_TAG}"
    workflow.subworkflows[0].model = model
    apply_planewave_cutoffs(workflow, ECUTWFC, ECUTRHO, unit_name=unit_name)
    apply_scf_kgrid(workflow, kgrid[name], material=materials_by_name[name], unit_name=unit_name)
    job = find_job_for_material(client, saved_materials[name]["_id"], workflow.name, ACCOUNT_ID,
                                statuses=("submitted", "queued", "active", "finished"))
    if job is not None:
        print(f"♻️  {name}: reusing existing {workflow_label} job {job['_id']}")
        return job["_id"], False
    job = create_job(
        api_client=client, materials=[saved_materials[name]], workflow=workflow, project_id=project_id,
        owner_id=ACCOUNT_ID, compute=compute.to_dict(), prefix=f"{workflow.name} {timestamp}",
    )
    print(f"✅ {name}: created {workflow_label} job {job['_id']}")
    return job["_id"], True


total_energy_workflow_config = WorkflowStandata.filter_by_application(application.name).get_by_name_first_match(
    TOTAL_ENERGY_SEARCH_TERM
)
energy_job_ids = {}
new_job_ids = []
for name in materials_by_name:
    energy_job_ids[name], is_new = find_or_create_job(name, total_energy_workflow_config, "Total Energy", SCF_UNIT)
    if is_new:
        new_job_ids.append(energy_job_ids[name])

In [ ]:
from mat3ra.notebooks_utils.api.job import submit_jobs, wait_for_jobs_to_finish_async

if new_job_ids:
    submit_jobs(client.jobs, new_job_ids)
    print(f"✅ Submitted {len(new_job_ids)} Total Energy job(s).")
if energy_job_ids:
    await wait_for_jobs_to_finish_async(client.jobs, list(energy_job_ids.values()), poll_interval=POLL_INTERVAL)

## 7. Relax the slabs (optional)

Runs only if `RELAX`: every atom of each slab relaxes at fixed cell, and the relaxed energy is the relaxation job's own.

In [ ]:
relax_job_ids = {}
new_relax_job_ids = []
if RELAX:
    relax_workflow_config = WorkflowStandata.filter_by_application(application.name).get_by_name_first_match(
        RELAX_WORKFLOW_SEARCH_TERM
    )
    for name in SLAB_NAMES.values():
        relax_job_ids[name], is_new = find_or_create_job(
            name, relax_workflow_config, "Fixed-cell Relaxation", RELAX_UNIT
        )
        if is_new:
            new_relax_job_ids.append(relax_job_ids[name])

if new_relax_job_ids:
    submit_jobs(client.jobs, new_relax_job_ids)
    print(f"✅ Submitted {len(new_relax_job_ids)} relaxation job(s).")
if relax_job_ids:
    await wait_for_jobs_to_finish_async(client.jobs, list(relax_job_ids.values()), poll_interval=POLL_INTERVAL)

## 8. Retrieve the results
### 8.1. Total energies

In [ ]:
from mat3ra.notebooks_utils.core.entity.property.api import get_properties_for_job

unrelaxed_energy = {name: get_properties_for_job(client, job_id, "total_energy")[0]["value"]
                    for name, job_id in energy_job_ids.items()}
relaxed_energy = {name: get_properties_for_job(client, job_id, "total_energy")[0]["value"]
                  for name, job_id in relax_job_ids.items()}
for name, energy in unrelaxed_energy.items():
    line = f"{name}: {energy:.4f} eV"
    if name in relaxed_energy:
        total_force = get_properties_for_job(client, relax_job_ids[name], "total_force")[0]
        line += (f", relaxed {relaxed_energy[name]:.4f} eV, "
                 f"residual force {total_force['value']:.4f} {total_force['units']}")
    print(line)

### 8.2. Cleavage, relaxation and surface energies

The slabs of a group are stoichiometric together: they contain $n$ bulk SrTiO3 units, and cleaving the crystal into
them creates two surfaces per slab. The paper divides that energy equally among those surfaces, Eqs. (1), (4) and (5):

$$E_{cleav} = \frac{1}{2 m}\left[\sum_{\Lambda} E^{unr}_{slab}(\Lambda) - n E_{bulk}\right],$$

with $m$ the number of slabs in the group (two for TiO + Sr and SrO + TiO2, one for O). Each slab then adds its own
relaxation energy, both faces relaxing, Eqs. (2) and (3):

$$E_{rel}(\Lambda) = \frac{1}{2}\left[E^{rel}_{slab}(\Lambda) - E^{unr}_{slab}(\Lambda)\right], \qquad
E_{surf}(\Lambda) = E_{cleav}(\Lambda) + E_{rel}(\Lambda).$$

The equal split is a convention; the sum of the two surface energies of a pair does not depend on it.

In [ ]:
bulk = materials_by_name[BULK_NAME]
bulk_energy_per_unit = unrelaxed_energy[BULK_NAME] / Counter(bulk.basis.elements.values)["Sr"]  # per SrTiO3 unit

cleavage_energy, relaxation_energy, surface_energy = {}, {}, {}
for group in CLEAVAGE_GROUPS:
    names = [SLAB_NAMES[label] for label in group]
    composition = sum((Counter(materials_by_name[name].basis.elements.values) for name in names), Counter())
    number_of_units = composition["Sr"]
    if composition != Counter({"Sr": number_of_units, "Ti": number_of_units, "O": 3 * number_of_units}):
        raise ValueError(f"{' + '.join(group)} is not stoichiometric: {dict(composition)}")
    slab_energy_sum = sum(unrelaxed_energy[name] for name in names)
    cleavage = (slab_energy_sum - number_of_units * bulk_energy_per_unit) / (2 * len(group))
    print(f"{' + '.join(group)}: {number_of_units} SrTiO3 units, cleavage energy {cleavage:.3f} eV per surface")
    for label, name in zip(group, names):
        cleavage_energy[label] = cleavage
        if RELAX:
            relaxation_energy[label] = (relaxed_energy[name] - unrelaxed_energy[name]) / 2
            surface_energy[label] = cleavage + relaxation_energy[label]

## 9. Comparison with Eglitis & Vanderbilt (2008)

Energies per 1x1 surface cell; the deviation is relative to the size of the paper's value, so for the (negative)
relaxation energies a positive deviation means less relaxation than the paper's.

In [ ]:
# eV per surface cell, Table VII: cleavage, relaxation and surface energy
EGLITIS = {"TiO": (4.61, -1.55, 3.06), "Sr": (4.61, -1.95, 2.66), "O": (3.36, -1.32, 2.04),
           "SrO": (1.39, -0.24, 1.15), "TiO2": (1.39, -0.16, 1.23)}


def format_comparison(ours, paper):
    """Our value, the paper's, and the deviation relative to the size of the paper's."""
    return f"{ours:.2f} (paper {paper:.2f}, {100 * (ours - paper) / abs(paper):+.1f} %)"


print(f"Ours: {MODEL_TAG}, {'relaxed' if RELAX else 'unrelaxed'}; paper: B3PW. Energies in eV.")
for label, cleavage in cleavage_energy.items():
    paper_cleavage, paper_relaxation, paper_surface = EGLITIS[label]
    line = f"{label:<5} cleavage {format_comparison(cleavage, paper_cleavage)}"
    if RELAX:
        line += (f"  relaxation {format_comparison(relaxation_energy[label], paper_relaxation)}"
                 f"  surface {format_comparison(surface_energy[label], paper_surface)}")
    print(line)
if RELAX and {"TiO", "Sr", "O"} <= surface_energy.keys():
    pair_sum = surface_energy["TiO"] + surface_energy["Sr"]
    print(f"TiO + Sr = {format_comparison(pair_sum, EGLITIS['TiO'][2] + EGLITIS['Sr'][2])}")
    terminations = ("TiO", "Sr", "O")
    print(f"(011), lowest first: {' < '.join(sorted(terminations, key=surface_energy.get))} "
          f"(paper {' < '.join(sorted(terminations, key=lambda label: EGLITIS[label][2]))})")

## References

[1] R. I. Eglitis and D. Vanderbilt, "First-principles calculations of atomic and electronic structure of SrTiO3 (001) and (011) surfaces", Phys. Rev. B 77, 195408 (2008). https://doi.org/10.1103/PhysRevB.77.195408